# 06 - Evaluacion final: test, calibracion, explicabilidad y veredicto de hipotesis

Rama: `05FinalModelChecks`. Este cuaderno cierra la Etapa 2 con el modelo ya elegido; **no busca ni entrena modelos nuevos** (salvo las curvas de aprendizaje, que reajustan B4 con fracciones de `train`).

**Lo que ya esta hecho (rama 04) y se baja de Kaggle en la celda 1:**

- Escalera B0 a B4 medida en `val` por escenario (A, B y C). B4 (HistGradientBoosting, trial 20) es el modelo principal.
- LightGBM no sustituye a B4; ablaciones por familia y H2 ya ejecutadas en `val`.
- Modelo final para la Etapa 3: B4 ajustado con el `train` de A, en `modelo_final/` (`ptcg_wp.predictor.PredictorVictoria`).
- Evaluacion comun para todos los modelos: `ptcg_wp.evaluation.metric_rows`.

**Lo que hace este cuaderno (pendientes de `05FinalModelChecks` en `PlanDeDesarrollo.md`):**

1. Evaluacion **unica** sobre `test` en A, B y C, global y por etapa, con intervalos por remuestreo de partidas.
2. Comparacion contra B0, B2 y la literatura (AUC 0.78-0.80).
3. Calibracion con `calib` (A y B): fiabilidad y Brier antes y despues.
4. Curvas de aprendizaje, analisis de errores y SHAP.
5. Prueba de H3 y veredicto de H1, H2 y H3.

Ejecuta las celdas de arriba abajo en un servidor de Colab con CPU + RAM alta.

In [ ]:
# PREPARACION COMPLETA DEL CUADERNO (rama 05FinalModelChecks)
# PROPOSITO: dejar este servidor listo con TODO lo producido hasta ahora:
# - Clona o actualiza el repo en la rama 05FinalModelChecks e instala las librerias fijadas
#   (y lightgbm 4.6.0, para que las pruebas de LightGBM tambien corran).
# - Muestra nucleos de CPU y RAM (se espera CPU + RAM alta: 8 nucleos, ~51 GiB).
# - Configura Kaggle y baja del dataset privado del equipo:
#   1) las 21 tablas a /content/datos (estados, partidas y particiones);
#   2) TODOS los resultados del entrenamiento a /content/resultados (B0 a B4, LightGBM,
#      ablaciones, H2, modelo final y READMEs), cada uno verificado con su SHA-256.
#   Lo que ya este en este servidor no se vuelve a bajar.
# ESPERADO: la linea de nucleos/RAM, "Tablas listas | caracteristicas: 62" y un resumen de
# resultados por carpeta, con "Todos los resultados del indice estan en /content/resultados".
import json
import os
import subprocess
import sys
import time
from collections import Counter
from pathlib import Path

REPO = "Adriel23456/Applied-AI-Agent-Project"
BRANCH = "05FinalModelChecks"
REPO_DIR = Path("/content/repo")
if not REPO_DIR.exists():
    !git clone -b {BRANCH} https://github.com/{REPO}.git /content/repo
else:
    !git -C /content/repo fetch origin {BRANCH}
    !git -C /content/repo checkout {BRANCH}
    !git -C /content/repo pull --ff-only origin {BRANCH}
%cd /content/repo
!pip install -q -r requirements.txt lightgbm==4.6.0
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
for nombre in [m for m in list(sys.modules) if m == "ptcg_wp" or m.startswith("ptcg_wp.")]:
    del sys.modules[nombre]

# Nucleos y RAM de este servidor, y credenciales de Kaggle
from ptcg_wp.colab import recursos, setup_kaggle
RECURSOS = recursos()
setup_kaggle()

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display
from sklearn import __version__ as sklearn_version
assert sklearn_version == "1.7.2", (f"Se requiere scikit-learn 1.7.2; version cargada: {sklearn_version}. "
                                    "Reinicia el runtime y ejecuta esta celda de nuevo.")
from ptcg_wp import store
from ptcg_wp.features import feature_columns
from ptcg_wp.ingest import load_days

# 1) Tablas procesadas -> /content/datos
DATA_DIR = store.DATOS_DIR
RESULTADOS = store.RESULTADOS_DIR
DATASET = store.load_dataset_id()
store.fetch(DATASET, DATA_DIR, store.expected_files(load_days("configs/dias.yaml")))
schema = pq.ParquetFile(sorted(DATA_DIR.glob("estados_*.parquet"))[0]).schema.names
FEATURES = feature_columns(schema)
assert len(FEATURES) == 62, len(FEATURES)

# 2) TODOS los resultados publicados (segun el indice del dataset) -> /content/resultados
indice = store.leer_indice(DATASET)
TODOS_LOS_RESULTADOS = sorted(r for r, fila in indice.items() if fila["tipo"] == "resultado")
assert store.listo(TODOS_LOS_RESULTADOS), "Hay resultados del indice que no se pudieron bajar"
por_carpeta = Counter("/".join(r.split("/")[:2]) if r.count("/") >= 2 else r.split("/")[0]
                      for r in TODOS_LOS_RESULTADOS)
display(pd.DataFrame(sorted(por_carpeta.items()), columns=["carpeta", "archivos"]))
print(f"Todos los resultados del indice estan en {RESULTADOS}: {len(TODOS_LOS_RESULTADOS)} archivos")

# Constantes y trazabilidad para el resto del cuaderno
SEED = 2026
SCENARIOS = ("A", "B", "C")
COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
DIRTY = bool(subprocess.check_output(["git", "-C", str(REPO_DIR), "status", "--porcelain"], text=True).strip())
RUN_INFO = {"commit": COMMIT, "working_tree_dirty": DIRTY, "seed": SEED,
            "fecha_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "recursos": RECURSOS}
print("Tablas listas | caracteristicas:", len(FEATURES), "| rama:", BRANCH, "| commit:", COMMIT[:7],
      "| cambios locales:", DIRTY)

In [ ]:
# PRUEBAS AUTOMATICAS DEL REPO
# PROPOSITO: confirmar que el codigo de esta rama esta sano ANTES de tocar test. Usan datos
# sinteticos y no descargan nada (unos 2 minutos). Como la celda anterior bajo todos los
# resultados e instalo lightgbm, aqui no se omite ninguna prueba.
# - Fuga y etiquetas, particiones, carga de datos y almacen de Kaggle.
# - Entrenamiento por lotes y protocolo agrupado de B2, B3 y B4.
# - Evaluacion comun igual a scikit-learn; H2 sin perdida de categorias; LightGBM sin avisos.
# - Modelo final: la observacion produce las mismas caracteristicas que el entrenamiento.
# - Evidencia de B2/B3 contra los resultados bajados en /content/resultados.
# ESPERADO: "86 passed" (en la rama 04 eran 86; si la rama 05 agrega pruebas, el numero sube).
# Si alguna dice "failed", NO seguir: avisar al equipo con el nombre de la prueba.
!python -m pytest -q tests

## Aqui empieza el trabajo de 05FinalModelChecks

Todo lo anterior solo prepara el servidor. Recomendaciones para seguir:

1. **Primero, congela el protocolo de `test` por escrito** en `docx/03Evidence/` (por ejemplo `DD-MM-26_ProtocoloEvaluacionFinal.md`), ANTES de calcular cualquier metrica en `test`. Debe decir que modelos se evaluan (B4 principal; B0, B1, B2 y B3 como referencia), como se calculan los intervalos (bootstrap por partida, 1000 remuestreos, percentil 95 %), como se calibra (isotonica ajustada solo con `calib`, en A y B, reportando antes y despues) y que resultado confirma o refuta H1, H2 (en la etapa `inicio`) y H3. `test` se usa **una sola vez**: ninguna decision puede tomarse despues de ver sus numeros.
2. **Usa las piezas que ya existen; no reimplementes:**
   - Metricas: `ptcg_wp.evaluation.metric_rows` (global y por etapa, cada partida con el mismo peso).
   - Intervalos emparejados de AUC: `ptcg_wp.model_comparison.paired_game_auc_bootstrap`.
   - Modelos: `/content/resultados/b0_b1/`, `b2/final/`, `b3/final/`, `b4/final/escenario_X/` y `modelo_final/` (este ultimo es el que usara la Etapa 3).
   - Lectura por lotes: `ptcg_wp.data.ScenarioStream`. Hoy `iter_batches` solo acepta `train` y `val` a proposito, como proteccion; habilitar `calib` y `test` es el primer cambio de codigo de esta rama y debe quedar documentado y con prueba.
3. **Mismo sistema que los cuadernos 02 a 05:** cada celda que calcula algo declara sus archivos y llama a `store.asegurar(archivos, calcular, mensaje)`: si ya estan en Kaggle los baja y no recalcula; si faltan, calcula y publica. Sugerencia de carpeta: `evaluacion_final/` (por ejemplo `evaluacion_final/test/escenario_A/metricas.csv`). Sin interruptores `True/False` y sin Google Drive.
4. **Orden sugerido:** protocolo -> calibracion en `calib` -> evaluacion unica en `test` con intervalos -> comparacion con B0, B2 y literatura -> H3 -> analisis de errores -> SHAP -> curvas de aprendizaje (lo mas caro: reentrenar B4 con 25/50/75 % de `train`) -> veredicto H1/H2/H3.
5. **Al cerrar cada pendiente,** marcalo como `[x]` en `docx/01SeguimientoDeTareas/PlanDeDesarrollo.md` con el resultado y donde quedo guardado, y registra el uso de IA en `docx/03Evidence/01UsoDeIA`.